# 特性

property(fget=None, fset=None, fdel=None, doc=None)

In [5]:
class LineItem:
    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price

    def get_weight(self):
        return self.weight

    def set_weight(self, value):
        if value > 0:
            self.weight = value
        else:
            raise ValueError("value must be > 0")

    weight = property(get_weight, set_weight)


li = LineItem("King", 23, 1.2)
print(li)

RecursionError: maximum recursion depth exceeded

In [24]:
class Class:
    data = "the class data attr"

    @property
    def prop(self):
        return "the prop value"


obj = Class()
print(vars(obj))
print(obj.data)
obj.data = "bar"
print(obj.data)
print(vars(obj))
print(Class.data)
print(Class.prop)
print(obj.prop)
# obj.prop = 'foo'
# print(obj.prop)
obj.__dict__["prop"] = "foo"
print(vars(obj))
print(obj.prop)
Class.prop = "baz"  # 销毁特性对象
print(obj.prop)
print(obj.data)
print(Class.data)
Class.data = property(lambda self: 'the "data" prop value')  # 增加类特性覆盖实例属性
print(obj.data)
del Class.data
print(obj.data)

{}
the class data attr
bar
{'data': 'bar'}
the class data attr
the prop value
{'data': 'bar', 'prop': 'foo'}
the prop value
foo
bar
the class data attr
the "data" prop value
bar


In [2]:
class Foo:
    @property
    def bar(self):
        """The bar attribute"""
        return self.__dict__["bar"]

    @bar.setter
    def bar(self, value):
        self.__dict__["bar"] = value


help(Foo.bar)

Help on property:

    The bar attribute



In [38]:
def quantity(storage_name):

    def qty_getter(instance):
        return instance.__dict__[storage_name]

    def qty_setter(instance, value):
        if value > 0:
            instance.__dict__[storage_name] = value
        else:
            raise ValueError("value must be > 0")

    return property(qty_getter, qty_setter)


class LineItem:
    weight = quantity("weight")
    price = quantity("price")

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


nutmeg = LineItem("Moluccan", 8, 13.96)
print(nutmeg.weight, nutmeg.price)
print(nutmeg.__dict__)
print(nutmeg.subtotal())
nutmeg.price = -2
print(nutmeg.price)

8 13.96
{'description': 'Moluccan', 'weight': 8, 'price': 13.96}
111.68


ValueError: value must be > 0

## 属性删除

In [33]:
class Deme:
    pass


d = Deme()
d.color = "green"

print(d.color)

del d.color
print(d.color)

green


AttributeError: 'Deme' object has no attribute 'color'

In [34]:
class BlackKnight:

    def __init__(self):
        self.phrases = [
            ("an arm", "'Tis but a scratch."),
            ("another arm", "It's just flesh wound"),
            ("a leg", "I'm invincible"),
            ("another leg", "All right, we'll call it a draw"),
        ]

    @property
    def member(self):
        print("next member is:")
        return self.phrases[0][0]

    @member.deleter
    def member(self):
        member, text = self.phrases.pop(0)
        print(f"BLACK KNIGHT (loses {member}) -- {text}")


knight = BlackKnight()
print(knight.member)
del knight.member
del knight.member
del knight.member

next member is:
an arm
BLACK KNIGHT (loses an arm) -- 'Tis but a scratch.
BLACK KNIGHT (loses another arm) -- It's just flesh wound
BLACK KNIGHT (loses a leg) -- I'm invincible


## 处理属性的重要属性和函数
- __class__ 对象所属类的引用
- __dict__ 存储对象或类的可写属性的映射
- __slots__ 类定义允许拥有的属性
- dir([object]) 列出对象的大多数属性 审查有或者没有__dir__属性的对象，不列出__dir__属性，但会列出其中的键。不列出几个特殊属性，如__mro__、__bases__、__name__。实现__dir__方法可以自定义dir函数输出。没有传参列出当前作用域中的名称。
- getattr(object, name [, default]) 获取属性值
- hasattr(object, name) 
- setattr(object, name, value) 可能创建新属性，也可能覆盖现有属性
- vars([object]) 获取对象的__dict__属性。如果对象定义了__slots__属性且实例没有__dict__属性，vars()函数不能处理，但dir()函数能处理。如果没有传参数，作用与locals()函数一样：返回本地作用域的字典。

In [ ]:
# dir()
# vars()
locals()

## 处理属性的特殊方法
使用点号表示法或内置函数getattr、hasattr、setattr会触发相应的特殊方法。但是直接通过实例的__dict__属性读写不会触发特殊方法。
对于用户自定义类，如果隐式调用特殊方法，仅当特殊方法在对象所属的类型上而不是在对象的实例字典中定义时，才能确保调用成功。
`特殊方法从类上获取，即便操作目标是实例也是如此`。特殊方法不被同名实例属性遮盖。

- __delattr__(self, name)
- __dir__(self)
- __getattr__(self, name) 仅当获取指定属性失败，搜索过obj、Class及其超类之后调用这个方法。
- __getattribute__(self, name) 直接获取指定名称的属性时始终调用这个方法。
- __setattr__(self, name, value) 尝试设置指定名称的属性时总会调用这个方法。

## 属性描述符

In [39]:
class Quantity:
    def __init__(self, storage_name):
        self.storage_name = storage_name

    def __set__(self, instance, value):
        if value > 0:
            instance.__dict__[self.storage_name] = value
        else:
            msg = f"{self.storage_name} must be > 0"
            raise ValueError(msg)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        else:
            return instance.__dict__[self.storage_name]


class LineItem:
    weight = Quantity("weight")
    price = Quantity("price")

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


li = LineItem("Apple", 12, 23.56)
print(li.subtotal())
li.price = -23
print(li.price)

282.71999999999997


ValueError: price must be > 0

In [42]:
class Quantity:
    def __set_name__(self, owner, name):
        self.storage_name = name

    def __set__(self, instance, value):
        if value > 0:
            instance.__dict__[self.storage_name] = value
        else:
            msg = f"{self.storage_name} must be > 0"
            raise ValueError(msg)


class LineItem:
    weight = Quantity()
    price = Quantity()

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


li = LineItem("Apple", 12, 23.56)
print(li.subtotal())
# li.price = -23
print(li.price, li.weight, li.description)

282.71999999999997
23.56 12 Apple


In [45]:
import abc


class Validated(abc.ABC):
    def __set_name__(self, owner, name):
        self.storage_name = name

    def __set__(self, instance, value):
        value = self.validate(self.storage_name, value)
        instance.__dict__[self.storage_name] = value

    @abc.abstractmethod
    def validate(self, name, value):
        """返回验证的值，或抛出ValueError"""


class Quantity(Validated):
    def validate(self, name, value):
        if value <= 0:
            msg = f"{self.storage_name} must be > 0"
            raise ValueError(msg)
        return value


class NonBank(Validated):
    def validate(self, name, value):
        value = value.strip()
        if not value:
            raise ValueError(f"{name} cannot be blank")
        return value


class LineItem:
    description = NonBank()
    price = Quantity()
    weight = Quantity()

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


li = LineItem("23", 12, 23.12)
print(li.subtotal())

277.44


In [5]:
def cls_name(obj_or_cls):
    cls = type(obj_or_cls)
    if cls is type:
        cls = obj_or_cls
    return cls.__name__.split(".")[-1]


def display(obj):
    cls = type(obj)
    if cls is type:
        return f"<class {obj.__name__}>"
    elif cls in [type(None), int]:
        return repr(obj)
    else:
        return f"<{cls_name(obj)}> object"


def print_args(name, *args):
    pseudo_args = ", ".join(display(x) for x in args)
    print(f"-> {cls_name(args[0])}.__{name}__({pseudo_args})")


class Overriding:
    def __get__(self, instance, owner):
        print_args("get", self, instance, owner)

    def __set__(self, instance, value):
        print_args("set", self, instance, value)


class OverridingNoGet:
    def __set__(self, instance, value):
        print_args("set", self, instance, value)


class NonOverriding:
    def __get__(self, instance, owner):
        print_args("get", self, instance, owner)


class Managed:
    over = Overriding()
    over_no_get = OverridingNoGet()
    non_over = NonOverriding()

    def spam(self):
        print(f"-> Managed.spam({display(self)})")


m = Managed()
m.spam()
m.over
m.over = 2
print(m.over)
m.__dict__["over"] = 3
vars(m)
m.over
# m.non_over
# m.over_no_get

-> Managed.spam(<Managed> object)
-> Overriding.__get__(<Overriding> object, <Managed> object, <class Managed>)
-> Overriding.__set__(<Overriding> object, <Managed> object, 2)
-> Overriding.__get__(<Overriding> object, <Managed> object, <class Managed>)
None
-> Overriding.__get__(<Overriding> object, <Managed> object, <class Managed>)


实现__set__方法的描述符属于覆盖型描述符，将覆盖对实例属性的赋值操作。
特性也是覆盖型描述符:如果没有提供设置值的函数，那么property类中的__set__方法就会抛出AttributeError异常。

In [8]:
import collections


class Text(collections.UserString):
    def __repr__(self):
        return "Text({!r})".format(self.data)

    def reverse(self):
        return self[::-1]


word = Text('forward')
print(word)
print(word.reverse())


forward
drawrof
